In [1]:
# 6 - MLP (copy this cell)
import pandas as pd, numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, classification_report

import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("processed_cardio_dataset.csv", sep=None, engine='python')
if 'id' in df.columns: df = df.drop('id', axis=1)
df = df.replace({True:1, False:0})
label_col = 'cardio'

X = df.drop(label_col, axis=1)
y = df[label_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Baseline MLP
mlp_base = MLPClassifier(max_iter=300, early_stopping=True, random_state=42)
mlp_base.fit(X_train_s, y_train)
yb = mlp_base.predict(X_test_s)
print("Baseline MLP Acc/F1:", accuracy_score(y_test, yb), f1_score(y_test, yb))

# Grid search for architectures
param_grid = {
    'hidden_layer_sizes': [(50,), (100,), (100,50)],
    'activation': ['relu','tanh'],
    'alpha': [1e-4, 1e-3]
}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
grid = GridSearchCV(MLPClassifier(max_iter=500, early_stopping=True, random_state=42), param_grid, cv=cv, scoring='f1', n_jobs=-1)
grid.fit(X_train_s, y_train)
best_mlp = grid.best_estimator_
print("Best MLP params:", grid.best_params_)

# Evaluate best
yp = best_mlp.predict(X_test_s)
ypr = best_mlp.predict_proba(X_test_s)[:,1]
print("Best MLP metrics:")
print("Accuracy:", accuracy_score(y_test, yp))
print("Precision:", precision_score(y_test, yp, zero_division=0))
print("Recall:", recall_score(y_test, yp, zero_division=0))
print("F1:", f1_score(y_test, yp, zero_division=0))
print("AUC:", roc_auc_score(y_test, ypr))
print("Confusion matrix:\n", confusion_matrix(y_test, yp))
print(classification_report(y_test, yp))

# Notes:
# - Use early_stopping to avoid long training times.
# - MLP benefits from scaling and hyperparameter tuning.


Baseline MLP Acc/F1: 0.7326545454545454 0.7266914498141264
Best MLP params: {'activation': 'relu', 'alpha': 0.001, 'hidden_layer_sizes': (50,)}
Best MLP metrics:
Accuracy: 0.7301090909090909
Precision: 0.7428571428571429
Recall: 0.6953710506980162
F1: 0.7183301707779887
AUC: 0.7940948536020132
Confusion matrix:
 [[5307 1638]
 [2073 4732]]
              precision    recall  f1-score   support

           0       0.72      0.76      0.74      6945
           1       0.74      0.70      0.72      6805

    accuracy                           0.73     13750
   macro avg       0.73      0.73      0.73     13750
weighted avg       0.73      0.73      0.73     13750

